# 04 · Modelo M3: DeBERTa-v3 con ajuste fino

**Entorno:** Kaggle con GPU (T4 ×2 o P100)

Modelo principal del proyecto. DeBERTa-v3 es un *Transformer* preentrenado que relaciona la explicación del estudiante con la pregunta y la respuesta seleccionada mediante atención. Este fue corrido desde la plataforma de Kaggle debido a su uso de GPU.

**Formulación:** plana, igual que M1. Clasificador de secuencias sobre la etiqueta conjunta `Category:Misconception`.

**Entrada:** `Question: … Answer: … Correct: yes/no Explanation: …`, construida con `src/preprocesamiento.py`.

### Cómo se corrió en Kaggle:

1. Crear un notebook nuevo en Kaggle y subir este archivo (*File → Import Notebook*).
2. En el panel derecho: **Accelerator → GPU T4 x2** (o P100) e **Internet → On**. Ambas opciones requieren tener la cuenta verificada con número de teléfono.
3. Antes, en la computadora: correr el notebook `02` y hacer `git push`, para que `data/splits/particion.csv` y las reglas estén en GitHub. Si el repositorio es privado, subirlo como *Dataset* de Kaggle y cambiar `RUTA_REPO_KAGGLE`.
4. *Run All*. Con la configuración por defecto tarda alrededor de una hora; para comprobar primero que todo funciona, poner `PRUEBA_RAPIDA = True` (unos minutos).
5. Descargar `salida_M3.zip` desde la pestaña *Output* y descomprimirlo en la raíz del repositorio. Contiene `models/m3_deberta/`, la predicción, el tuning y `results/tiempos.csv`.

**Salidas:** `models/m3_deberta/` (fuera de Git), `results/tuning/tuning_M3_deberta.csv`, `results/predicciones/pred_M3_deberta.csv` y la fila de M3 en `results/tiempos.csv`.

## 1. Entorno

In [72]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JuanDsm04/ds-project2.git"
RUTA_REPO_KAGGLE = Path("/kaggle/working/ds-project2")   # o la ruta del Dataset si el repo es privado
EN_KAGGLE = Path("/kaggle").exists()

if EN_KAGGLE:
    if not RUTA_REPO_KAGGLE.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(RUTA_REPO_KAGGLE)], check=True)
    os.chdir(RUTA_REPO_KAGGLE)

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: /kaggle/working/ds-project2


In [73]:
import json
import shutil
import time
import inspect

import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

from src import preprocesamiento as pp
from src.metricas import resumen_metricas, top_k
from src.utilidades import guardar_tuning, registrar_tiempo

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

torch 2.11.0+cu128 | transformers 5.16.1
GPU: Tesla T4


## 2. Configuración

Los hiperparámetros a explorar siguen la propuesta de la etapa de investigación: tasa de aprendizaje en {2e-5, 3e-5}, de 3 a 4 épocas con *early stopping*, *batch* de 16, *warmup* del 10 % y *weight decay* de 0.01. Además se prueba la entropía cruzada ponderada por clase (pesos ∝ 1/√frecuencia, con tope) para el desbalance.

In [74]:
MODELO_BASE = os.environ.get("M3_MODELO_BASE", "microsoft/deberta-v3-base")
PRUEBA_RAPIDA = os.environ.get("M3_PRUEBA_RAPIDA", "0") == "1"   # True: subconjunto pequeño y 1 época

NOMBRE = "M3_deberta"
SEMILLA = 42
EPOCAS = 1 if PRUEBA_RAPIDA else 4
BATCH = 16
WARMUP = 0.1
WEIGHT_DECAY = 0.01
PACIENCIA = 1            # épocas sin mejora antes de detener
TOPE_PESO = 10.0         # peso máximo de una clase en la pérdida ponderada

# Configuraciones a probar: (tasa de aprendizaje, pérdida ponderada)
CONFIGURACIONES = [(2e-5, False), (3e-5, False), ("mejor", True)]
if PRUEBA_RAPIDA:
    CONFIGURACIONES = [(3e-5, False), (3e-5, True)]

DIR_RUNS = RAIZ / "runs_m3"
DIR_SALIDA = pp.DIR_MODELOS / "m3_deberta"
USAR_FP16 = torch.cuda.is_available()
set_seed(SEMILLA)

## 3. Datos

In [75]:
df = pp.cargar_train_clean()
if not pp.RUTA_PARTICION.exists() or not pp.RUTA_RESPUESTAS.exists():
    # Respaldo: la partición es determinística, así que se puede regenerar igual que en el notebook 02
    print("No se encontró la partición; se regenera con la semilla 42.")
    pp.RUTA_PARTICION.parent.mkdir(parents=True, exist_ok=True)
    pp.generar_particion(df).to_csv(pp.RUTA_PARTICION, index=False)
    pp.guardar_reglas(pp.aplicar_particion(df)["train"], df)

datos = pp.aplicar_particion(df)
respuestas, preguntas, misc_por_pregunta = pp.cargar_reglas()
train = pp.preparar_dataframe(datos["train"], respuestas, preguntas)
val = pp.preparar_dataframe(datos["val"], respuestas, preguntas)

if PRUEBA_RAPIDA:
    train = train.sample(800, random_state=SEMILLA).reset_index(drop=True)
    val = val.sample(300, random_state=SEMILLA).reset_index(drop=True)

clases = sorted(datos["train"]["target"].unique())   # todas las etiquetas del split de entrenamiento completo
etiqueta_a_id = {c: i for i, c in enumerate(clases)}
y_train = train["target"].map(etiqueta_a_id).values
y_val = val["target"].map(etiqueta_a_id).values
print(f"train={len(train):,} | val={len(val):,} | clases={len(clases)}")
train["texto_entrada"].iloc[0]

train=25,688 | val=3,668 | clases=64


'Question: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them are shaded.] Answer: 1/3 Correct: yes Explanation: 1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.'

## 4. Tokenización y longitud máxima

In [76]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
longitudes = np.array([len(ids) for ids in tokenizer(train["texto_entrada"].tolist())["input_ids"]])
print(pd.Series(longitudes).describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(1).to_string())

MAX_LEN = int(min(256, np.ceil(np.percentile(longitudes, 99.9) / 16) * 16))
print("MAX_LEN =", MAX_LEN, f"| filas truncadas en train: {(longitudes > MAX_LEN).mean():.3%}")

count    25688.0
mean        53.2
std         16.8
min         22.0
50%         51.0
90%         76.0
99%         96.0
99.9%      114.0
max        250.0
MAX_LEN = 128 | filas truncadas en train: 0.016%


In [77]:
class DatasetTexto(Dataset):
    """Textos ya tokenizados (sin relleno); el relleno se hace por lote."""
    def __init__(self, textos, etiquetas):
        self.enc = tokenizer(list(textos), truncation=True, max_length=MAX_LEN)
        self.etiquetas = etiquetas
    def __len__(self):
        return len(self.etiquetas)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.etiquetas[i])
        return item

ds_train = DatasetTexto(train["texto_entrada"], y_train)
ds_val = DatasetTexto(val["texto_entrada"], y_val)
collator = DataCollatorWithPadding(tokenizer)

## 5. Pérdida ponderada y métricas

- **Pesos de clase:** `(n_max / n_c)^0.5`, con tope de 10, para que las clases raras pesen más sin desestabilizar el entrenamiento.
- **Métricas en validación:** macro-F1, exactitud y MAP@3, sin y con la máscara de etiquetas. El mejor punto de control se elige por macro-F1 con máscara.

In [78]:
conteo = np.bincount(y_train, minlength=len(clases)).astype(float)
pesos_clase = np.clip(np.sqrt(conteo.max() / np.maximum(conteo, 1)), 1.0, TOPE_PESO)
mascara_val = pp.mascara_etiquetas(clases, val["QuestionId"], val["es_correcta"], misc_por_pregunta)
clases_np = np.array(clases)

class TrainerPonderado(Trainer):
    def __init__(self, *args, pesos=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pesos = None if pesos is None else torch.tensor(pesos, dtype=torch.float)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        etiquetas = inputs.pop("labels")
        salidas = model(**inputs)
        pesos = None if self.pesos is None else self.pesos.to(salidas.logits.device)
        perdida = torch.nn.functional.cross_entropy(salidas.logits.float(), etiquetas, weight=pesos)
        return (perdida, salidas) if return_outputs else perdida

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    proba = softmax(logits)
    reales = clases_np[etiquetas]
    sin = resumen_metricas(reales, top_k(proba, clases_np)[0])
    con = proba * mascara_val[: len(proba)]
    con = con / con.sum(axis=1, keepdims=True)
    con = resumen_metricas(reales, top_k(con, clases_np)[0])
    return {**{f"{k}_sin_mascara": v for k, v in sin.items()}, **con}

# Compatibilidad: el argumento cambió de nombre en versiones recientes de transformers
PARAM_EVAL = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"

## 6. Ajuste de hiperparámetros

Cada configuración se entrena con *early stopping* sobre validación. Solo se conserva en disco el mejor modelo encontrado hasta el momento.

In [79]:
resultados = []
mejor_f1, mejor_cfg = -1.0, None

def construir_argumentos(dir_run, lr):
    # Compatible con transformers 4.x y 5.x: el warmup se calcula en pasos (warmup_ratio ya
    # no existe en versiones recientes) y se omiten los argumentos que la versión no reconozca.
    n_gpu = max(1, torch.cuda.device_count())
    pasos_por_epoca = int(np.ceil(len(ds_train) / (BATCH * n_gpu)))
    deseados = dict(
        output_dir=str(dir_run),
        learning_rate=lr,
        num_train_epochs=EPOCAS,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH * 4,
        warmup_steps=int(WARMUP * pasos_por_epoca * EPOCAS),
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="linear",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=USAR_FP16,
        group_by_length=True,
        logging_steps=100,
        report_to="none",
        seed=SEMILLA,
        **{PARAM_EVAL: "epoch"},
    )
    validos = inspect.signature(TrainingArguments).parameters
    omitidos = [k for k in deseados if k not in validos]
    if omitidos:
        print("Argumentos no soportados por esta versión de transformers (se omiten):", omitidos)
    return TrainingArguments(**{k: v for k, v in deseados.items() if k in validos})

def entrenar(lr, ponderada):
    dir_run = DIR_RUNS / f"lr{lr}_pond{int(ponderada)}"
    args = construir_argumentos(dir_run, lr)

    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=len(clases),
        id2label=dict(enumerate(clases)), label2id=etiqueta_a_id,
        ignore_mismatched_sizes=True,   # la cabeza de clasificación se crea nueva con 64 salidas
    ).float()   # los pesos de DeBERTa-v3 vienen en fp16; el entrenamiento mixto necesita el modelo en float32

    
    trainer = TrainerPonderado(
        model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
        data_collator=collator, compute_metrics=calcular_metricas,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PACIENCIA)],
        pesos=pesos_clase if ponderada else None,
    )
    inicio = time.time()
    trainer.train()
    duracion = time.time() - inicio
    metricas = trainer.evaluate()
    return trainer, metricas, duracion, dir_run

for lr, ponderada in CONFIGURACIONES:
    if lr == "mejor":
        lr = mejor_cfg["learning_rate"]
    trainer, met, duracion, dir_run = entrenar(lr, ponderada)
    fila = {
        "learning_rate": lr, "perdida_ponderada": ponderada, "epocas_max": EPOCAS,
        "epoca_mejor": trainer.state.best_model_checkpoint, "max_len": MAX_LEN,
        "f1_macro": met["eval_f1_macro"], "accuracy": met["eval_accuracy"], "map_at_3": met["eval_map_at_3"],
        "f1_macro_sin_mascara": met["eval_f1_macro_sin_mascara"], "map_at_3_sin_mascara": met["eval_map_at_3_sin_mascara"],
        "segundos_entrenamiento": round(duracion, 1),
    }
    resultados.append(fila)
    print(f"lr={lr:.0e} | ponderada={ponderada} | F1 macro={fila['f1_macro']:.4f} | MAP@3={fila['map_at_3']:.4f} | {duracion / 60:.1f} min")

    if fila["f1_macro"] > mejor_f1:
        mejor_f1, mejor_cfg = fila["f1_macro"], fila
        if DIR_SALIDA.exists():
            shutil.rmtree(DIR_SALIDA)
        trainer.save_model(str(DIR_SALIDA))
        tokenizer.save_pretrained(str(DIR_SALIDA))
    del trainer
    shutil.rmtree(dir_run, ignore_errors=True)
    torch.cuda.empty_cache()

Argumentos no soportados por esta versión de transformers (se omiten): ['group_by_length']


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,F1 Macro Sin Mascara,Accuracy Sin Mascara,Map At 3 Sin Mascara,F1 Macro,Accuracy,Map At 3
1,0.848260,0.772540,0.232137,0.762541,0.850600,0.257520,0.768266,0.868548
2,0.567421,0.557880,0.347314,0.824973,0.898264,0.348340,0.824973,0.901218
3,0.452973,0.486862,0.441631,0.840240,0.909169,0.442571,0.840240,0.910714
4,0.363855,0.473807,0.460432,0.852236,0.915894,0.483047,0.853599,0.917848


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro Sin Mascara,Accuracy Sin Mascara,Map At 3 Sin Mascara,F1 Macro,Accuracy,Map At 3
0.377599,0.598608,4,0.583034,0.845147,0.914940,0.583253,0.845420,0.916258


lr=3e-05 | ponderada=True | F1 macro=0.5833 | MAP@3=0.9163 | 27.8 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [80]:
tabla = pd.DataFrame(resultados)
tabla["epoca_mejor"] = tabla["epoca_mejor"].astype(str).str.extract(r"checkpoint-(\d+)")[0]
tabla = tabla.rename(columns={"epoca_mejor": "paso_mejor_checkpoint"})
guardar_tuning(tabla, NOMBRE)
tabla.round({c: 4 for c in tabla.columns if c.startswith(("f1", "accuracy", "map"))})

,learning_rate,perdida_ponderada,epocas_max,paso_mejor_checkpoint,max_len,f1_macro,accuracy,map_at_3,f1_macro_sin_mascara,map_at_3_sin_mascara,segundos_entrenamiento
0,0.00002,False,4,3212,128,0.4830,0.8536,0.9178,0.4604,0.9159,1664.6
1,0.00003,False,4,3212,128,0.5121,0.8582,0.9228,0.5123,0.9219,1664.7
2,0.00003,True,4,3212,128,0.5833,0.8454,0.9163,0.5830,0.9149,1665.7


## 7. Guardado del modelo final

Se guarda la configuración que necesita `src/prediccion.py`: el orden de las clases, la longitud máxima y si conviene aplicar la máscara (se aplica si no empeora el macro-F1 de validación).

In [81]:
usar_mascara = bool(mejor_cfg["f1_macro"] >= mejor_cfg["f1_macro_sin_mascara"])
config_proyecto = {
    "modelo": NOMBRE,
    "modelo_base": MODELO_BASE,
    "clases": clases,
    "max_len": MAX_LEN,
    "usar_mascara": usar_mascara,
    "hiperparametros": {"learning_rate": mejor_cfg["learning_rate"], "perdida_ponderada": bool(mejor_cfg["perdida_ponderada"]),
                        "epocas_max": EPOCAS, "batch": BATCH, "warmup": WARMUP, "weight_decay": WEIGHT_DECAY},
    "metricas_validacion": {k: float(mejor_cfg[k]) for k in ["f1_macro", "accuracy", "map_at_3"]},
    "prueba_rapida": PRUEBA_RAPIDA,
}
with open(DIR_SALIDA / "config_proyecto.json", "w", encoding="utf-8") as f:
    json.dump(config_proyecto, f, ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in config_proyecto.items() if k != "clases"}, indent=2))

{
  "modelo": "M3_deberta",
  "modelo_base": "microsoft/deberta-v3-base",
  "max_len": 128,
  "usar_mascara": true,
  "hiperparametros": {
    "learning_rate": 3e-05,
    "perdida_ponderada": true,
    "epocas_max": 4,
    "batch": 16,
    "warmup": 0.1,
    "weight_decay": 0.01
  },
  "metricas_validacion": {
    "f1_macro": 0.5832533404387313,
    "accuracy": 0.8454198473282443,
    "map_at_3": 0.9162577244638312
  },
  "prueba_rapida": false
}


## 8. Predicciones sobre el conjunto de prueba

Con `src.prediccion.predecir_lote`, la misma función que usa la aplicación.

In [82]:
from src import prediccion
from src.utilidades import guardar_predicciones

test_df = datos["test"].sample(300, random_state=SEMILLA) if PRUEBA_RAPIDA else datos["test"]
inicio = time.time()
pred_test = prediccion.predecir_lote(NOMBRE, test_df)
t_inferencia = time.time() - inicio

ruta_pred = guardar_predicciones(pred_test, NOMBRE)
registrar_tiempo(NOMBRE, mejor_cfg["segundos_entrenamiento"], t_inferencia)
print(f"{len(pred_test):,} predicciones en {t_inferencia:.1f} s -> {ruta_pred.relative_to(RAIZ)}")
pred_test.head()

7,340 predicciones en 50.9 s -> results/predicciones/pred_M3_deberta.csv


,row_id,top1,top2,top3,prob1,prob2,prob3
0,0,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.711661,0.135103,0.098017
1,16,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.711259,0.135226,0.098257
2,19,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.711830,0.135219,0.097874
3,24,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.716989,0.131944,0.096391
4,26,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.716972,0.132498,0.096447


In [83]:
# Verificación: predecir_lote reproduce la métrica de validación del mejor modelo
pred_val = prediccion.predecir_lote(NOMBRE, val)
resumen_metricas(val["target"], pred_val[["top1", "top2", "top3"]].values)

{'f1_macro': 0.029443580635074754,
 'accuracy': 0.5815158124318429,
 'map_at_3': 0.7231006906579426}

In [84]:
prediccion.predecir(NOMBRE, 31772, r"\( \frac{3}{6} \)", "there are 3 white triangles and 6 blue ones so its 3/6")

[{'etiqueta': 'False_Neither:NA',
  'category': 'False_Neither',
  'misconception': 'NA',
  'prob': 0.4248420000076294},
 {'etiqueta': 'False_Misconception:Incomplete',
  'category': 'False_Misconception',
  'misconception': 'Incomplete',
  'prob': 0.28162598609924316},
 {'etiqueta': 'False_Correct:NA',
  'category': 'False_Correct',
  'misconception': 'NA',
  'prob': 0.16349899768829346}]

## 9. Empaquetar resultados para descargar

Descargar `salida_M3.zip` (pestaña *Output* de Kaggle) y descomprimirlo en la raíz del repositorio local.

In [85]:
import zipfile

ruta_zip = (Path("/kaggle/working") if EN_KAGGLE else RAIZ) / "salida_M3.zip"
archivos = [p for p in DIR_SALIDA.rglob("*") if p.is_file()] + [
    RAIZ / "results/predicciones/pred_M3_deberta.csv",
    RAIZ / "results/tuning/tuning_M3_deberta.csv",
    RAIZ / "results/tiempos.csv",
]
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in archivos:
        z.write(p, p.relative_to(RAIZ))
shutil.rmtree(DIR_RUNS, ignore_errors=True)
print(f"{ruta_zip} ({ruta_zip.stat().st_size / 1e6:.0f} MB)")

/kaggle/working/salida_M3.zip (581 MB)
